In [603]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler

from xgboost import XGBRegressor

from combine_features import read_data

# Experiments

In [69]:
df = read_data("../rhea-soil-nutrient-prediction-challenge/Train.csv")
df.head()

,ID,Longitude,Latitude,Depth_cm,ph,Area,Cropland nitrogen per unit area,Cropland phosphorus per unit area,Cropland potassium per unit area,tmin_avg,...,Cu,Fe,Mg,Mn,N,P,K,Na,S,Zn
0,BF9XTB,37.65189,-3.15440,20-50,6.405,Kenya,14.202173,3.902236,-7.263155,15.240530,...,5.826,81.780,306.836,270.240,0.79,NaN,300.951,NaN,NaN,NaN
1,2RWYTR,37.63612,-3.08585,20-50,6.419,Kenya,14.202173,3.902236,-7.263155,15.240530,...,4.346,97.198,407.980,185.557,1.11,NaN,292.696,NaN,NaN,NaN
2,XZI9Q6,39.55580,-2.67218,20-50,8.388,Kenya,14.202173,3.902236,-7.263155,21.969696,...,3.657,42.672,1256.319,178.299,0.45,NaN,814.911,NaN,NaN,NaN
3,4CBCVY,39.55477,-2.67196,20-50,8.302,Kenya,14.202173,3.902236,-7.263155,21.969696,...,3.376,52.861,1322.732,464.137,0.31,NaN,815.337,NaN,NaN,NaN
4,F9GK9S,39.55477,-2.67196,20-50,8.292,Kenya,14.202173,3.902236,-7.263155,21.969696,...,3.351,46.057,1134.898,274.565,0.45,NaN,928.238,NaN,NaN,NaN


In [70]:
df.iloc[:, 40:].info()

<class 'pandas.DataFrame'>
RangeIndex: 13454 entries, 0 to 13453
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Ca         13454 non-null  float64
 1   C_organic  13454 non-null  float64
 2   C_total    13416 non-null  float64
 3   Cu         13416 non-null  float64
 4   Fe         13416 non-null  float64
 5   Mg         13454 non-null  float64
 6   Mn         13416 non-null  float64
 7   N          13416 non-null  float64
 8   P          0 non-null      float64
 9   K          13454 non-null  float64
 10  Na         38 non-null     float64
 11  S          0 non-null      float64
 12  Zn         0 non-null      float64
dtypes: float64(13)
memory usage: 1.3 MB


In [71]:
pred_to_keep = pd.read_csv("../rhea-soil-nutrient-prediction-challenge/TargetPred_To_Keep.csv")
pred_to_keep.head()

,ID,Al,B,Ca,Cu,Fe,K,Mg,Mn,N,Na,P,S,Zn
0,00A83S,1,1,1,1,1,1,1,1,1,1,1,1,1
1,00F2Q3,1,1,1,1,1,1,1,1,1,1,1,1,1
2,00FNFP,1,0,1,1,1,1,1,1,1,0,0,0,0
3,01MFSS,1,0,1,1,1,1,1,1,1,0,0,0,0
4,02851F,1,0,1,1,1,1,1,1,1,0,0,0,0


In [72]:
pred_to_keep.sum()

ID    00A83S00F2Q300FNFP01MFSS02851F02C3Q502I9O502L2...
Al                                                 6070
B                                                  1345
Ca                                                 6070
Cu                                                 6065
Fe                                                 6065
K                                                  6070
Mg                                                 6070
Mn                                                 6065
N                                                  6065
Na                                                 1350
P                                                  1345
S                                                  1345
Zn                                                 1345
dtype: object

In [73]:
encoder = LabelEncoder()
df['Depth'] = encoder.fit_transform(df['Depth_cm'])
# df.drop(columns=['ID', 'Area', 'Depth_cm'], inplace=True)
# df.dropna(axis=1, thresh=int(0.8 * df.shape[0]), inplace=True) # drops target columns :(
# df.dropna(axis=0, inplace=True)
df.head()

,ID,Longitude,Latitude,Depth_cm,ph,Area,Cropland nitrogen per unit area,Cropland phosphorus per unit area,Cropland potassium per unit area,tmin_avg,...,Fe,Mg,Mn,N,P,K,Na,S,Zn,Depth
0,BF9XTB,37.65189,-3.15440,20-50,6.405,Kenya,14.202173,3.902236,-7.263155,15.240530,...,81.780,306.836,270.240,0.79,NaN,300.951,NaN,NaN,NaN,1
1,2RWYTR,37.63612,-3.08585,20-50,6.419,Kenya,14.202173,3.902236,-7.263155,15.240530,...,97.198,407.980,185.557,1.11,NaN,292.696,NaN,NaN,NaN,1
2,XZI9Q6,39.55580,-2.67218,20-50,8.388,Kenya,14.202173,3.902236,-7.263155,21.969696,...,42.672,1256.319,178.299,0.45,NaN,814.911,NaN,NaN,NaN,1
3,4CBCVY,39.55477,-2.67196,20-50,8.302,Kenya,14.202173,3.902236,-7.263155,21.969696,...,52.861,1322.732,464.137,0.31,NaN,815.337,NaN,NaN,NaN,1
4,F9GK9S,39.55477,-2.67196,20-50,8.292,Kenya,14.202173,3.902236,-7.263155,21.969696,...,46.057,1134.898,274.565,0.45,NaN,928.238,NaN,NaN,NaN,1


In [74]:
target = [
        "Al",
        "B",
        "Ca",
        "C_organic",
        "C_total",
        "Cu",
        "Fe",
        "K",
        "Mg",
        "Mn",
        "N",
        "Na",
        "P",
        "S",
        "Zn",
    ]
x = df.drop(columns=target, errors='ignore')
x = x.drop(columns=['ID', 'Area', 'Depth_cm', 'ph'], errors='ignore')
y_columns = df.columns.difference(x.columns)
y_columns = [col for col in y_columns if col not in ["C_organic", "C_total", "Area", "Depth_cm", "ID"]]
y = df[y_columns]
y.fillna(0, inplace=True)
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [75]:
x_train.head()

,Longitude,Latitude,Cropland nitrogen per unit area,Cropland phosphorus per unit area,Cropland potassium per unit area,tmin_avg,tmax_avg,prec_avg,Bananas,"Beans, dry",...,Rice,"Seed cotton, unginned",Sesame seed,Sorghum,Soya beans,Sugar cane,Sweet potatoes,Tomatoes,Unmanufactured tobacco,Depth
4721,-2.19999,7.20972,-2.917073,-2.796745,-18.097209,22.215910,30.598484,103.590550,10390.800000,1165.118182,...,2719.254545,900.927273,0.000000,1169.454545,1634.036364,25018.527273,1781.009091,7377.254545,413.118182,0
7732,-11.01100,14.76583,3.931445,-2.550627,-3.758418,23.316288,37.174244,54.082947,34266.136364,811.044444,...,3123.272727,985.490909,420.227273,1037.981818,620.363636,72963.909091,17452.418182,17080.290909,2304.281818,0
5746,-1.07758,7.64100,-2.917073,-2.796745,-18.097209,22.660984,32.393940,103.886536,10390.800000,1165.118182,...,2719.254545,900.927273,0.000000,1169.454545,1634.036364,25018.527273,1781.009091,7377.254545,413.118182,0
12178,14.92311,-16.19789,9.219964,0.017264,-9.459173,15.484848,30.342804,61.349070,24064.809091,405.136364,...,1022.400000,1446.963636,263.136364,227.736364,603.709091,38876.818182,8256.936364,10544.109091,1018.445455,0
12952,15.22342,-12.29724,9.219964,0.017264,-9.459173,13.121212,25.611742,121.739590,24064.809091,405.136364,...,1022.400000,1446.963636,263.136364,227.736364,603.709091,38876.818182,8256.936364,10544.109091,1018.445455,0


In [76]:
y_train.head()

,Al,B,Ca,Cu,Fe,K,Mg,Mn,N,Na,P,S,Zn,ph
4721,668.664,0.0,2440.023,3.406,141.737,138.646,286.859,337.732,1.90,0.0,0.0,0.0,0.0,6.969
7732,981.937,0.0,2336.159,2.910,159.714,271.761,671.278,93.069,0.56,0.0,0.0,0.0,0.0,6.699
5746,837.742,0.0,2120.437,2.700,109.035,218.283,327.699,250.103,1.91,0.0,0.0,0.0,0.0,6.509
12178,730.772,0.0,740.481,0.966,172.359,87.640,134.619,102.362,0.80,0.0,0.0,0.0,0.0,6.161
12952,1145.199,0.0,541.405,2.794,83.008,160.257,149.018,69.130,0.62,0.0,0.0,0.0,0.0,5.903


In [77]:
models = {}
for col in y_columns:
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(x_train, y_train[col])
    y_pred = model.predict(x_test)
    rmse = root_mean_squared_error(y_test[col], y_pred)
    models[col] = model
    print(f"{col}: RMSE = {rmse}")

Al: RMSE = 130.2396468091458
B: RMSE = 0.0
Ca: RMSE = 959.4224269660466
Cu: RMSE = 0.7452205663210932
Fe: RMSE = 29.148475487969623
K: RMSE = 100.47139795477366
Mg: RMSE = 134.07210263822418
Mn: RMSE = 43.330061762577444
N: RMSE = 0.46218505583420955
Na: RMSE = 1.4646935033365307
P: RMSE = 0.0
S: RMSE = 0.0
Zn: RMSE = 0.0
ph: RMSE = 0.34927910272145685


In [78]:
for col, model in models.items():
    fi = pd.DataFrame(model.feature_importances_, index=x_train.columns, columns=['Feature Importance'])
    fi.sort_values(by='Feature Importance', ascending=False, inplace=True)
    print(f"Feature importance for {col}:")
    print(fi.head(10))

Feature importance for Al:
                                   Feature Importance
prec_avg                                     0.536625
Longitude                                    0.152047
Latitude                                     0.119135
tmin_avg                                     0.052261
Soya beans                                   0.046809
tmax_avg                                     0.022330
Pineapples                                   0.020283
Depth                                        0.012970
Coffee, green                                0.005833
Cropland phosphorus per unit area            0.004214
Feature importance for B:
                                                 Feature Importance
Longitude                                                       0.0
Seed cotton, unginned                                           0.0
Other fruits, n.e.c.                                            0.0
Other pulses n.e.c.                                             0.0
Other vegetab

# Generating test answers

## Importing df

In [1021]:
train = pd.read_csv("../rhea-soil-nutrient-prediction-challenge/Train.csv")
train.drop(columns='ph', inplace=True)
train.head()

,ID,Longitude,Latitude,start_date,end_date,horizon_lower,horizon_upper,Depth_cm,Al,B,...,electrical_conductivity,Fe,Mg,Mn,N,P,K,Na,S,Zn
0,O2TONM,35.18756,-8.62390,01/01/2008,31/12/2018,50,20,20-50,1109.856,NaN,...,NaN,92.366,200.601,107.257,2.24,NaN,283.103,NaN,NaN,NaN
1,BQLUK6,35.18558,-8.62300,01/01/2008,31/12/2018,50,20,20-50,1168.364,NaN,...,NaN,115.923,197.771,90.005,1.57,NaN,215.459,NaN,NaN,NaN
2,LSET8M,35.18579,-8.62221,01/01/2008,31/12/2018,50,20,20-50,1137.113,NaN,...,NaN,78.709,188.114,120.433,1.02,NaN,398.656,NaN,NaN,NaN
3,LEEL7I,35.18266,-8.62177,01/01/2008,31/12/2018,50,20,20-50,1117.349,NaN,...,NaN,127.527,156.417,112.036,1.12,NaN,267.354,NaN,NaN,NaN
4,LDNGO2,35.12984,-8.62005,01/01/2008,31/12/2018,50,20,20-50,1219.203,NaN,...,NaN,77.542,114.809,57.906,1.19,NaN,229.682,NaN,NaN,NaN


In [1022]:
test_df = pd.read_csv("../rhea-soil-nutrient-prediction-challenge/TestSet.csv")
test_df.head()

,ID,Latitude,Longitude,Depth_cm,Target_Al,Target_B,Target_Ca,Target_Cu,Target_Fe,Target_K,Target_Mg,Target_Mn,Target_N,Target_Na,Target_P,Target_S,Target_Zn
0,8ZMJRO,-0.746,37.094,0-20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,DCC6DM,-0.785,37.178,0-20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,T50LK1,-0.629,37.126,0-20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,FNLYT0,-0.351,35.308,0-20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,FP5E12,-1.894,36.987,0-20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [1023]:
pred_to_keep = pd.read_csv("../rhea-soil-nutrient-prediction-challenge/TargetPred_To_Keep.csv")
pred_to_keep.head()

,ID,Al,B,Ca,Cu,Fe,K,Mg,Mn,N,Na,P,S,Zn
0,00A83S,1,1,1,1,1,1,1,1,1,1,1,1,1
1,00F2Q3,1,1,1,1,1,1,1,1,1,1,1,1,1
2,00FNFP,1,0,1,1,1,1,1,1,1,0,0,0,0
3,01MFSS,1,0,1,1,1,1,1,1,1,0,0,0,0
4,02851F,1,0,1,1,1,1,1,1,1,0,0,0,0


In [1024]:
olek_features = pd.read_csv("../features/olek-features.csv")
print(olek_features.shape)
olek_features.head()

(50368, 59)


,ID,Country,Area,tmin_avg,tmax_avg,prec_avg,B11_x,B12_x,B2,B3,...,Fe_pred,K_pred,Mg_pred,N_pred,P_pred,S_pred,Zn_pred,carbon,clay,ph
0,O2TONM,Tanzania,Tanzania,12.727273,22.628788,123.26878,1287.0,699.0,243.0,364.0,...,28.964100,133.289780,120.510418,1.208738e+07,8.974182,2.320117,0.822119,32.0,36.0,5.6
1,BQLUK6,Tanzania,Tanzania,12.727273,22.628788,123.26878,3215.0,2426.0,789.0,1066.0,...,26.112639,133.289780,108.947172,2.979958e+06,7.166170,2.004166,0.648721,30.0,40.0,5.7
2,LSET8M,Tanzania,Tanzania,12.727273,22.628788,123.26878,3127.0,2220.0,697.0,943.0,...,28.964100,133.289780,120.510418,1.634984e+06,7.166170,2.004166,0.648721,30.0,39.0,5.7
3,LEEL7I,Tanzania,Tanzania,12.727273,22.628788,123.26878,2085.0,1370.0,499.0,691.0,...,26.112639,108.947172,133.289780,2.979958e+06,7.166170,1.718282,0.822119,31.0,40.0,5.5
4,LDNGO2,Tanzania,Tanzania,12.674242,23.077652,110.35551,2059.0,1076.0,324.0,436.0,...,23.532530,147.413159,98.484316,1.997196e+06,7.166170,2.004166,0.648721,33.0,39.0,5.0


In [1025]:
olek_features.columns

Index(['ID', 'Country', 'Area', 'tmin_avg', 'tmax_avg', 'prec_avg', 'B11_x',
       'B12_x', 'B2', 'B3', 'B4', 'B8', 'aspect', 'elevation', 'hillshade',
       'slope', 'cec_0-5cm_mean', 'cec_15-30cm_mean', 'cec_30-60cm_mean',
       'cec_5-15cm_mean', 'clay_0-5cm_mean', 'clay_15-30cm_mean',
       'clay_30-60cm_mean', 'clay_5-15cm_mean', 'phh2o_0-5cm_mean',
       'phh2o_15-30cm_mean', 'phh2o_30-60cm_mean', 'phh2o_5-15cm_mean',
       'sand_0-5cm_mean', 'sand_15-30cm_mean', 'sand_30-60cm_mean',
       'sand_5-15cm_mean', 'soc_0-5cm_mean', 'soc_15-30cm_mean',
       'soc_30-60cm_mean', 'soc_5-15cm_mean', 'B04', 'B05', 'B06', 'B07',
       'B08', 'B09', 'B10', 'B11_y', 'B12_y', 'B13', 'B14', 'Al_pred',
       'Ca_pred', 'Fe_pred', 'K_pred', 'Mg_pred', 'N_pred', 'P_pred', 'S_pred',
       'Zn_pred', 'carbon', 'clay', 'ph'],
      dtype='str')

In [1026]:
olek_features.drop(columns=['B11_x', 'B12_x', 'B2', 'B3', 'B4', 'B8'], inplace=True, errors='ignore')
olek_features.columns

Index(['ID', 'Country', 'Area', 'tmin_avg', 'tmax_avg', 'prec_avg', 'aspect',
       'elevation', 'hillshade', 'slope', 'cec_0-5cm_mean', 'cec_15-30cm_mean',
       'cec_30-60cm_mean', 'cec_5-15cm_mean', 'clay_0-5cm_mean',
       'clay_15-30cm_mean', 'clay_30-60cm_mean', 'clay_5-15cm_mean',
       'phh2o_0-5cm_mean', 'phh2o_15-30cm_mean', 'phh2o_30-60cm_mean',
       'phh2o_5-15cm_mean', 'sand_0-5cm_mean', 'sand_15-30cm_mean',
       'sand_30-60cm_mean', 'sand_5-15cm_mean', 'soc_0-5cm_mean',
       'soc_15-30cm_mean', 'soc_30-60cm_mean', 'soc_5-15cm_mean', 'B04', 'B05',
       'B06', 'B07', 'B08', 'B09', 'B10', 'B11_y', 'B12_y', 'B13', 'B14',
       'Al_pred', 'Ca_pred', 'Fe_pred', 'K_pred', 'Mg_pred', 'N_pred',
       'P_pred', 'S_pred', 'Zn_pred', 'carbon', 'clay', 'ph'],
      dtype='str')

## Combining data

In [1027]:
#landforms_df = pd.read_csv('../features/landforms_features.csv')
#landforms_df.head()

In [1028]:
#landforms_df.drop(columns = 'Unnamed: 0', inplace=True)
#landforms_df.head()

In [1029]:
features = olek_features.copy()
features.drop(columns=['Country', 'Area'], inplace=True)
features.head()

,ID,tmin_avg,tmax_avg,prec_avg,aspect,elevation,hillshade,slope,cec_0-5cm_mean,cec_15-30cm_mean,...,Fe_pred,K_pred,Mg_pred,N_pred,P_pred,S_pred,Zn_pred,carbon,clay,ph
0,O2TONM,12.727273,22.628788,123.26878,192,1895,189,13,107.0,73.0,...,28.964100,133.289780,120.510418,1.208738e+07,8.974182,2.320117,0.822119,32.0,36.0,5.6
1,BQLUK6,12.727273,22.628788,123.26878,265,1884,215,11,107.0,72.0,...,26.112639,133.289780,108.947172,2.979958e+06,7.166170,2.004166,0.648721,30.0,40.0,5.7
2,LSET8M,12.727273,22.628788,123.26878,315,1888,195,7,107.0,72.0,...,28.964100,133.289780,120.510418,1.634984e+06,7.166170,2.004166,0.648721,30.0,39.0,5.7
3,LEEL7I,12.727273,22.628788,123.26878,45,1898,163,8,104.0,69.0,...,26.112639,108.947172,133.289780,2.979958e+06,7.166170,1.718282,0.822119,31.0,40.0,5.5
4,LDNGO2,12.674242,23.077652,110.35551,278,1894,201,7,99.0,69.0,...,23.532530,147.413159,98.484316,1.997196e+06,7.166170,2.004166,0.648721,33.0,39.0,5.0


In [1030]:
features['ratio_Ca_Mg'] = features['Ca_pred'] / (features['Mg_pred'] + 0.001)
features['ratio_Fe_Al'] = features['Fe_pred'] / (features['Al_pred'] + 0.001)
features['CIA_proxy'] = features['Al_pred'] / (features['Al_pred'] + features['Ca_pred'] + features['K_pred'] + 0.001)

In [1031]:
features.head()

,ID,tmin_avg,tmax_avg,prec_avg,aspect,elevation,hillshade,slope,cec_0-5cm_mean,cec_15-30cm_mean,...,N_pred,P_pred,S_pred,Zn_pred,carbon,clay,ph,ratio_Ca_Mg,ratio_Fe_Al,CIA_proxy
0,O2TONM,12.727273,22.628788,123.26878,192,1895,189,13,107.0,73.0,...,1.208738e+07,8.974182,2.320117,0.822119,32.0,36.0,5.6,4.985793,0.107502,0.268470
1,BQLUK6,12.727273,22.628788,123.26878,265,1884,215,11,107.0,72.0,...,2.979958e+06,7.166170,2.004166,0.648721,30.0,40.0,5.7,4.989271,0.096919,0.284719
2,LSET8M,12.727273,22.628788,123.26878,315,1888,195,7,107.0,72.0,...,1.634984e+06,7.166170,2.004166,0.648721,30.0,39.0,5.7,4.080518,0.118855,0.280515
3,LEEL7I,12.727273,22.628788,123.26878,45,1898,163,8,104.0,69.0,...,2.979958e+06,7.166170,1.718282,0.822119,31.0,40.0,5.5,3.019180,0.096919,0.345063
4,LDNGO2,12.674242,23.077652,110.35551,278,1894,201,7,99.0,69.0,...,1.997196e+06,7.166170,2.004166,0.648721,33.0,39.0,5.0,3.024485,0.087343,0.376974


In [1032]:
train = pd.merge(train, features, on='ID', how='left')
train.head()

,ID,Longitude,Latitude,start_date,end_date,horizon_lower,horizon_upper,Depth_cm,Al,B,...,N_pred,P_pred,S_pred,Zn_pred,carbon,clay,ph,ratio_Ca_Mg,ratio_Fe_Al,CIA_proxy
0,O2TONM,35.18756,-8.62390,01/01/2008,31/12/2018,50,20,20-50,1109.856,NaN,...,1.208738e+07,8.974182,2.320117,0.822119,32.0,36.0,5.6,4.985793,0.107502,0.268470
1,BQLUK6,35.18558,-8.62300,01/01/2008,31/12/2018,50,20,20-50,1168.364,NaN,...,2.979958e+06,7.166170,2.004166,0.648721,30.0,40.0,5.7,4.989271,0.096919,0.284719
2,LSET8M,35.18579,-8.62221,01/01/2008,31/12/2018,50,20,20-50,1137.113,NaN,...,1.634984e+06,7.166170,2.004166,0.648721,30.0,39.0,5.7,4.080518,0.118855,0.280515
3,LEEL7I,35.18266,-8.62177,01/01/2008,31/12/2018,50,20,20-50,1117.349,NaN,...,2.979958e+06,7.166170,1.718282,0.822119,31.0,40.0,5.5,3.019180,0.096919,0.345063
4,LDNGO2,35.12984,-8.62005,01/01/2008,31/12/2018,50,20,20-50,1219.203,NaN,...,1.997196e+06,7.166170,2.004166,0.648721,33.0,39.0,5.0,3.024485,0.087343,0.376974


In [1033]:
test = pd.merge(test_df, features, on='ID', how='left')
test.head()

,ID,Latitude,Longitude,Depth_cm,Target_Al,Target_B,Target_Ca,Target_Cu,Target_Fe,Target_K,...,N_pred,P_pred,S_pred,Zn_pred,carbon,clay,ph,ratio_Ca_Mg,ratio_Fe_Al,CIA_proxy
0,8ZMJRO,-0.746,37.094,0-20,0.0,0.0,0.0,0.0,0.0,0.0,...,2.434201e+07,10.023176,8.025013,8.025013,38.0,42.0,5.9,2.722545,0.199472,0.187717
1,DCC6DM,-0.785,37.178,0-20,0.0,0.0,0.0,0.0,0.0,0.0,...,8.954293e+06,12.463738,7.166170,4.473947,35.0,43.0,5.6,3.330628,0.329843,0.197227
2,T50LK1,-0.629,37.126,0-20,0.0,0.0,0.0,0.0,0.0,0.0,...,5.987314e+07,12.463738,8.974182,3.953032,33.0,37.0,5.6,3.681391,0.270322,0.252583
3,FNLYT0,-0.351,35.308,0-20,0.0,0.0,0.0,0.0,0.0,0.0,...,5.403639e+08,12.463738,3.481689,3.953032,36.0,39.0,5.5,4.974933,0.365532,0.191087
4,FP5E12,-1.894,36.987,0-20,0.0,0.0,0.0,0.0,0.0,0.0,...,8.114058e+05,17.174145,3.953032,1.718282,28.0,29.0,6.4,4.066525,0.493167,0.099152


In [1034]:
#train = pd.merge(train, landforms_df, on='ID', how='left')
#train.head()

In [1035]:
#test = pd.merge(test, landforms_df, on='ID', how='left')
#test.head()

In [1036]:
encoder = LabelEncoder()
train['Depth'] = encoder.fit_transform(train['Depth_cm'])
test['Depth'] = encoder.transform(test['Depth_cm'])
train.drop(columns=['ID', 'Area', 'Depth_cm'], inplace=True, errors='ignore')
test.drop(columns=['ID', 'Depth_cm'], inplace=True, errors='ignore')
test.head()

,Latitude,Longitude,Target_Al,Target_B,Target_Ca,Target_Cu,Target_Fe,Target_K,Target_Mg,Target_Mn,...,P_pred,S_pred,Zn_pred,carbon,clay,ph,ratio_Ca_Mg,ratio_Fe_Al,CIA_proxy,Depth
0,-0.746,37.094,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,10.023176,8.025013,8.025013,38.0,42.0,5.9,2.722545,0.199472,0.187717,0
1,-0.785,37.178,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,12.463738,7.166170,4.473947,35.0,43.0,5.6,3.330628,0.329843,0.197227,0
2,-0.629,37.126,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,12.463738,8.974182,3.953032,33.0,37.0,5.6,3.681391,0.270322,0.252583,0
3,-0.351,35.308,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,12.463738,3.481689,3.953032,36.0,39.0,5.5,4.974933,0.365532,0.191087,0
4,-1.894,36.987,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,17.174145,3.953032,1.718282,28.0,29.0,6.4,4.066525,0.493167,0.099152,0


In [1037]:
train["Na"] = train["Na"].fillna(train["Na"].mean(), inplace=False)
train = train.fillna(0, inplace=False)
train.head()

,Longitude,Latitude,start_date,end_date,horizon_lower,horizon_upper,Al,B,Ca,C_organic,...,P_pred,S_pred,Zn_pred,carbon,clay,ph,ratio_Ca_Mg,ratio_Fe_Al,CIA_proxy,Depth
0,35.18756,-8.62390,01/01/2008,31/12/2018,50,20,1109.856,0.0,1535.388,30.66,...,8.974182,2.320117,0.822119,32.0,36.0,5.6,4.985793,0.107502,0.268470,1
1,35.18558,-8.62300,01/01/2008,31/12/2018,50,20,1168.364,0.0,751.408,21.15,...,7.166170,2.004166,0.648721,30.0,40.0,5.7,4.989271,0.096919,0.284719,1
2,35.18579,-8.62221,01/01/2008,31/12/2018,50,20,1137.113,0.0,468.391,15.64,...,7.166170,2.004166,0.648721,30.0,39.0,5.7,4.080518,0.118855,0.280515,1
3,35.18266,-8.62177,01/01/2008,31/12/2018,50,20,1117.349,0.0,739.698,15.63,...,7.166170,1.718282,0.822119,31.0,40.0,5.5,3.019180,0.096919,0.345063,1
4,35.12984,-8.62005,01/01/2008,31/12/2018,50,20,1219.203,0.0,240.071,18.49,...,7.166170,2.004166,0.648721,33.0,39.0,5.0,3.024485,0.087343,0.376974,1


In [1038]:
test.head()

,Latitude,Longitude,Target_Al,Target_B,Target_Ca,Target_Cu,Target_Fe,Target_K,Target_Mg,Target_Mn,...,P_pred,S_pred,Zn_pred,carbon,clay,ph,ratio_Ca_Mg,ratio_Fe_Al,CIA_proxy,Depth
0,-0.746,37.094,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,10.023176,8.025013,8.025013,38.0,42.0,5.9,2.722545,0.199472,0.187717,0
1,-0.785,37.178,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,12.463738,7.166170,4.473947,35.0,43.0,5.6,3.330628,0.329843,0.197227,0
2,-0.629,37.126,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,12.463738,8.974182,3.953032,33.0,37.0,5.6,3.681391,0.270322,0.252583,0
3,-0.351,35.308,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,12.463738,3.481689,3.953032,36.0,39.0,5.5,4.974933,0.365532,0.191087,0
4,-1.894,36.987,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,17.174145,3.953032,1.718282,28.0,29.0,6.4,4.066525,0.493167,0.099152,0


## Separating Data

In [1039]:
target = [
        "Al",
        "B",
        "Ca",
        "C_organic",
        "C_total",
        "Cu",
        "Fe",
        "K",
        "Mg",
        "Mn",
        "N",
        "Na",
        "P",
        "S",
        "Zn",
    ]
X = train.drop(columns=target, errors='ignore')
y_columns = train.columns.difference(X.columns)
y_columns = [col for col in y_columns if col not in ["C_organic", "C_total", "Area", "Depth_cm", "ID"]]
y = train[y_columns]


In [1040]:
X.head()

,Longitude,Latitude,start_date,end_date,horizon_lower,horizon_upper,electrical_conductivity,tmin_avg,tmax_avg,prec_avg,...,P_pred,S_pred,Zn_pred,carbon,clay,ph,ratio_Ca_Mg,ratio_Fe_Al,CIA_proxy,Depth
0,35.18756,-8.62390,01/01/2008,31/12/2018,50,20,0.0,12.727273,22.628788,123.26878,...,8.974182,2.320117,0.822119,32.0,36.0,5.6,4.985793,0.107502,0.268470,1
1,35.18558,-8.62300,01/01/2008,31/12/2018,50,20,0.0,12.727273,22.628788,123.26878,...,7.166170,2.004166,0.648721,30.0,40.0,5.7,4.989271,0.096919,0.284719,1
2,35.18579,-8.62221,01/01/2008,31/12/2018,50,20,0.0,12.727273,22.628788,123.26878,...,7.166170,2.004166,0.648721,30.0,39.0,5.7,4.080518,0.118855,0.280515,1
3,35.18266,-8.62177,01/01/2008,31/12/2018,50,20,0.0,12.727273,22.628788,123.26878,...,7.166170,1.718282,0.822119,31.0,40.0,5.5,3.019180,0.096919,0.345063,1
4,35.12984,-8.62005,01/01/2008,31/12/2018,50,20,0.0,12.674242,23.077652,110.35551,...,7.166170,2.004166,0.648721,33.0,39.0,5.0,3.024485,0.087343,0.376974,1


In [1041]:
y.head()

,Al,B,Ca,Cu,Fe,K,Mg,Mn,N,Na,P,S,Zn
0,1109.856,0.0,1535.388,2.259,92.366,283.103,200.601,107.257,2.24,37.98841,0.0,0.0,0.0
1,1168.364,0.0,751.408,1.822,115.923,215.459,197.771,90.005,1.57,37.98841,0.0,0.0,0.0
2,1137.113,0.0,468.391,1.913,78.709,398.656,188.114,120.433,1.02,37.98841,0.0,0.0,0.0
3,1117.349,0.0,739.698,2.876,127.527,267.354,156.417,112.036,1.12,37.98841,0.0,0.0,0.0
4,1219.203,0.0,240.071,1.825,77.542,229.682,114.809,57.906,1.19,37.98841,0.0,0.0,0.0


In [1042]:
test.head()

,Latitude,Longitude,Target_Al,Target_B,Target_Ca,Target_Cu,Target_Fe,Target_K,Target_Mg,Target_Mn,...,P_pred,S_pred,Zn_pred,carbon,clay,ph,ratio_Ca_Mg,ratio_Fe_Al,CIA_proxy,Depth
0,-0.746,37.094,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,10.023176,8.025013,8.025013,38.0,42.0,5.9,2.722545,0.199472,0.187717,0
1,-0.785,37.178,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,12.463738,7.166170,4.473947,35.0,43.0,5.6,3.330628,0.329843,0.197227,0
2,-0.629,37.126,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,12.463738,8.974182,3.953032,33.0,37.0,5.6,3.681391,0.270322,0.252583,0
3,-0.351,35.308,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,12.463738,3.481689,3.953032,36.0,39.0,5.5,4.974933,0.365532,0.191087,0
4,-1.894,36.987,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,17.174145,3.953032,1.718282,28.0,29.0,6.4,4.066525,0.493167,0.099152,0


In [1043]:
target_pred = [
    "Target_Al",
    "Target_B",
    "Target_Ca",
    "Target_Cu",
    "Target_Fe",
    "Target_K",
    "Target_Mg",
    "Target_Mn",
    "Target_N",
    "Target_Na",
    "Target_P",
    "Target_S",
    "Target_Zn",
]

In [1044]:
X_pred = test.drop(columns=target_pred, errors='ignore')
y_pred_columns = test.columns.difference(X_pred.columns)
y_pred_columns = [col for col in y_pred_columns if col not in ["C_organic", "C_total", "Area", "Depth_cm", "ID"]]
y_pred = test[y_pred_columns]

In [1045]:
X_pred.head()

,Latitude,Longitude,tmin_avg,tmax_avg,prec_avg,aspect,elevation,hillshade,slope,cec_0-5cm_mean,...,P_pred,S_pred,Zn_pred,carbon,clay,ph,ratio_Ca_Mg,ratio_Fe_Al,CIA_proxy,Depth
0,-0.746,37.094,14.543561,25.986742,108.980880,63,1402,169,4,185.0,...,10.023176,8.025013,8.025013,38.0,42.0,5.9,2.722545,0.199472,0.187717,0
1,-0.785,37.178,15.577652,27.297348,91.997140,39,1191,142,20,212.0,...,12.463738,7.166170,4.473947,35.0,43.0,5.6,3.330628,0.329843,0.197227,0
2,-0.629,37.126,14.102273,25.522728,115.549255,298,1405,244,24,236.0,...,12.463738,8.974182,3.953032,33.0,37.0,5.6,3.681391,0.270322,0.252583,0
3,-0.351,35.308,11.918561,24.070076,165.598710,173,2041,177,7,194.0,...,12.463738,3.481689,3.953032,36.0,39.0,5.5,4.974933,0.365532,0.191087,0
4,-1.894,36.987,14.017045,25.181818,45.469700,315,1552,192,5,206.0,...,17.174145,3.953032,1.718282,28.0,29.0,6.4,4.066525,0.493167,0.099152,0


In [1046]:
y_pred.head()

,Target_Al,Target_B,Target_Ca,Target_Cu,Target_Fe,Target_K,Target_Mg,Target_Mn,Target_N,Target_Na,Target_P,Target_S,Target_Zn
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [1047]:
diff = set(X.columns) - set(X_pred.columns)
print(diff)
X.drop(columns=diff, inplace=True)

{'end_date', 'start_date', 'horizon_upper', 'electrical_conductivity', 'horizon_lower'}


In [1048]:
X_pred = X_pred[X.columns]
X_pred.head()

,Longitude,Latitude,tmin_avg,tmax_avg,prec_avg,aspect,elevation,hillshade,slope,cec_0-5cm_mean,...,P_pred,S_pred,Zn_pred,carbon,clay,ph,ratio_Ca_Mg,ratio_Fe_Al,CIA_proxy,Depth
0,37.094,-0.746,14.543561,25.986742,108.980880,63,1402,169,4,185.0,...,10.023176,8.025013,8.025013,38.0,42.0,5.9,2.722545,0.199472,0.187717,0
1,37.178,-0.785,15.577652,27.297348,91.997140,39,1191,142,20,212.0,...,12.463738,7.166170,4.473947,35.0,43.0,5.6,3.330628,0.329843,0.197227,0
2,37.126,-0.629,14.102273,25.522728,115.549255,298,1405,244,24,236.0,...,12.463738,8.974182,3.953032,33.0,37.0,5.6,3.681391,0.270322,0.252583,0
3,35.308,-0.351,11.918561,24.070076,165.598710,173,2041,177,7,194.0,...,12.463738,3.481689,3.953032,36.0,39.0,5.5,4.974933,0.365532,0.191087,0
4,36.987,-1.894,14.017045,25.181818,45.469700,315,1552,192,5,206.0,...,17.174145,3.953032,1.718282,28.0,29.0,6.4,4.066525,0.493167,0.099152,0


In [1049]:
ph = X['ph']
X = X.drop(columns=['ph'], errors='ignore')
X.head()

,Longitude,Latitude,tmin_avg,tmax_avg,prec_avg,aspect,elevation,hillshade,slope,cec_0-5cm_mean,...,N_pred,P_pred,S_pred,Zn_pred,carbon,clay,ratio_Ca_Mg,ratio_Fe_Al,CIA_proxy,Depth
0,35.18756,-8.62390,12.727273,22.628788,123.26878,192,1895,189,13,107.0,...,1.208738e+07,8.974182,2.320117,0.822119,32.0,36.0,4.985793,0.107502,0.268470,1
1,35.18558,-8.62300,12.727273,22.628788,123.26878,265,1884,215,11,107.0,...,2.979958e+06,7.166170,2.004166,0.648721,30.0,40.0,4.989271,0.096919,0.284719,1
2,35.18579,-8.62221,12.727273,22.628788,123.26878,315,1888,195,7,107.0,...,1.634984e+06,7.166170,2.004166,0.648721,30.0,39.0,4.080518,0.118855,0.280515,1
3,35.18266,-8.62177,12.727273,22.628788,123.26878,45,1898,163,8,104.0,...,2.979958e+06,7.166170,1.718282,0.822119,31.0,40.0,3.019180,0.096919,0.345063,1
4,35.12984,-8.62005,12.674242,23.077652,110.35551,278,1894,201,7,99.0,...,1.997196e+06,7.166170,2.004166,0.648721,33.0,39.0,3.024485,0.087343,0.376974,1


In [1050]:
X_pred = X_pred[X.columns]
X_pred.head()

,Longitude,Latitude,tmin_avg,tmax_avg,prec_avg,aspect,elevation,hillshade,slope,cec_0-5cm_mean,...,N_pred,P_pred,S_pred,Zn_pred,carbon,clay,ratio_Ca_Mg,ratio_Fe_Al,CIA_proxy,Depth
0,37.094,-0.746,14.543561,25.986742,108.980880,63,1402,169,4,185.0,...,2.434201e+07,10.023176,8.025013,8.025013,38.0,42.0,2.722545,0.199472,0.187717,0
1,37.178,-0.785,15.577652,27.297348,91.997140,39,1191,142,20,212.0,...,8.954293e+06,12.463738,7.166170,4.473947,35.0,43.0,3.330628,0.329843,0.197227,0
2,37.126,-0.629,14.102273,25.522728,115.549255,298,1405,244,24,236.0,...,5.987314e+07,12.463738,8.974182,3.953032,33.0,37.0,3.681391,0.270322,0.252583,0
3,35.308,-0.351,11.918561,24.070076,165.598710,173,2041,177,7,194.0,...,5.403639e+08,12.463738,3.481689,3.953032,36.0,39.0,4.974933,0.365532,0.191087,0
4,36.987,-1.894,14.017045,25.181818,45.469700,315,1552,192,5,206.0,...,8.114058e+05,17.174145,3.953032,1.718282,28.0,29.0,4.066525,0.493167,0.099152,0


## Additional prep for model

In [1051]:
X.columns

Index(['Longitude', 'Latitude', 'tmin_avg', 'tmax_avg', 'prec_avg', 'aspect',
       'elevation', 'hillshade', 'slope', 'cec_0-5cm_mean', 'cec_15-30cm_mean',
       'cec_30-60cm_mean', 'cec_5-15cm_mean', 'clay_0-5cm_mean',
       'clay_15-30cm_mean', 'clay_30-60cm_mean', 'clay_5-15cm_mean',
       'phh2o_0-5cm_mean', 'phh2o_15-30cm_mean', 'phh2o_30-60cm_mean',
       'phh2o_5-15cm_mean', 'sand_0-5cm_mean', 'sand_15-30cm_mean',
       'sand_30-60cm_mean', 'sand_5-15cm_mean', 'soc_0-5cm_mean',
       'soc_15-30cm_mean', 'soc_30-60cm_mean', 'soc_5-15cm_mean', 'B04', 'B05',
       'B06', 'B07', 'B08', 'B09', 'B10', 'B11_y', 'B12_y', 'B13', 'B14',
       'Al_pred', 'Ca_pred', 'Fe_pred', 'K_pred', 'Mg_pred', 'N_pred',
       'P_pred', 'S_pred', 'Zn_pred', 'carbon', 'clay', 'ratio_Ca_Mg',
       'ratio_Fe_Al', 'CIA_proxy', 'Depth'],
      dtype='str')

In [1052]:
features_to_drop = ['Longitude', 'Latitude', 'Cropland nitrogen per unit area',
       'Cropland phosphorus per unit area', 'Cropland potassium per unit area', 'Bananas', 'Beans, dry',
       'Cashew nuts, in shell', 'Cassava, fresh',
       'Chillies and peppers, green (Capsicum spp. and Pimenta spp.)',
       'Coffee, green', 'Groundnuts, excluding shelled', 'Maize (corn)',
       'Mangoes, guavas and mangosteens', 'Millet',
       'Onions and shallots, dry (excluding dehydrated)', 'Oranges',
       'Other fruits, n.e.c.', 'Other pulses n.e.c.',
       'Other vegetables, fresh n.e.c.', 'Pineapples', 'Potatoes', 'Rice',
       'Seed cotton, unginned', 'Sesame seed', 'Sorghum', 'Soya beans',
       'Sugar cane', 'Sweet potatoes', 'Tomatoes', 'Unmanufactured tobacco',
       'Depth']

In [1053]:
X.drop(columns=features_to_drop, inplace=True, errors='ignore')
X_pred.drop(columns=features_to_drop, inplace=True, errors='ignore')

In [1054]:
check = pd.read_csv("../rhea-soil-nutrient-prediction-challenge/Train.csv")
check.head()

,ID,Longitude,Latitude,start_date,end_date,horizon_lower,horizon_upper,Depth_cm,Al,B,...,Fe,Mg,Mn,N,ph,P,K,Na,S,Zn
0,O2TONM,35.18756,-8.62390,01/01/2008,31/12/2018,50,20,20-50,1109.856,NaN,...,92.366,200.601,107.257,2.24,5.942,NaN,283.103,NaN,NaN,NaN
1,BQLUK6,35.18558,-8.62300,01/01/2008,31/12/2018,50,20,20-50,1168.364,NaN,...,115.923,197.771,90.005,1.57,5.722,NaN,215.459,NaN,NaN,NaN
2,LSET8M,35.18579,-8.62221,01/01/2008,31/12/2018,50,20,20-50,1137.113,NaN,...,78.709,188.114,120.433,1.02,5.510,NaN,398.656,NaN,NaN,NaN
3,LEEL7I,35.18266,-8.62177,01/01/2008,31/12/2018,50,20,20-50,1117.349,NaN,...,127.527,156.417,112.036,1.12,5.817,NaN,267.354,NaN,NaN,NaN
4,LDNGO2,35.12984,-8.62005,01/01/2008,31/12/2018,50,20,20-50,1219.203,NaN,...,77.542,114.809,57.906,1.19,4.980,NaN,229.682,NaN,NaN,NaN


In [1055]:
check.iloc[:,5:].info()

<class 'pandas.DataFrame'>
RangeIndex: 44298 entries, 0 to 44297
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   horizon_lower            44298 non-null  int64  
 1   horizon_upper            44298 non-null  int64  
 2   Depth_cm                 44298 non-null  str    
 3   Al                       44296 non-null  float64
 4   B                        1909 non-null   float64
 5   Ca                       44298 non-null  float64
 6   C_organic                44298 non-null  float64
 7   C_total                  42348 non-null  float64
 8   Cu                       44257 non-null  float64
 9   electrical_conductivity  1907 non-null   float64
 10  Fe                       44257 non-null  float64
 11  Mg                       44298 non-null  float64
 12  Mn                       44255 non-null  float64
 13  N                        44257 non-null  float64
 14  ph                       44295 no

## Standarization

In [1056]:
#columns_landform = [col for col in X.columns if col.startswith('landform_')]
numeric_cols = X.select_dtypes(include=[np.number]).columns
#numeric_cols = [col for col in numeric_cols if col not in columns_landform]
numeric_cols

Index(['tmin_avg', 'tmax_avg', 'prec_avg', 'aspect', 'elevation', 'hillshade',
       'slope', 'cec_0-5cm_mean', 'cec_15-30cm_mean', 'cec_30-60cm_mean',
       'cec_5-15cm_mean', 'clay_0-5cm_mean', 'clay_15-30cm_mean',
       'clay_30-60cm_mean', 'clay_5-15cm_mean', 'phh2o_0-5cm_mean',
       'phh2o_15-30cm_mean', 'phh2o_30-60cm_mean', 'phh2o_5-15cm_mean',
       'sand_0-5cm_mean', 'sand_15-30cm_mean', 'sand_30-60cm_mean',
       'sand_5-15cm_mean', 'soc_0-5cm_mean', 'soc_15-30cm_mean',
       'soc_30-60cm_mean', 'soc_5-15cm_mean', 'B04', 'B05', 'B06', 'B07',
       'B08', 'B09', 'B10', 'B11_y', 'B12_y', 'B13', 'B14', 'Al_pred',
       'Ca_pred', 'Fe_pred', 'K_pred', 'Mg_pred', 'N_pred', 'P_pred', 'S_pred',
       'Zn_pred', 'carbon', 'clay', 'ratio_Ca_Mg', 'ratio_Fe_Al', 'CIA_proxy'],
      dtype='str')

In [1057]:
scaler = StandardScaler()
X[numeric_cols] = scaler.fit_transform(X[numeric_cols])
X_pred[numeric_cols] = scaler.transform(X_pred[numeric_cols])
X.head()

,tmin_avg,tmax_avg,prec_avg,aspect,elevation,hillshade,slope,cec_0-5cm_mean,cec_15-30cm_mean,cec_30-60cm_mean,...,Mg_pred,N_pred,P_pred,S_pred,Zn_pred,carbon,clay,ratio_Ca_Mg,ratio_Fe_Al,CIA_proxy
0,-1.201331,-1.831983,1.410737,0.252200,1.511893,0.658543,1.975520,-0.741007,-0.967002,-0.988639,...,-0.610901,-0.021464,0.480508,-0.604010,-0.890036,1.012217,1.245357,0.201224,-0.965492,0.501095
1,-1.201331,-1.831983,1.410737,0.934989,1.492899,2.566571,1.524698,-0.741007,-0.980205,-0.988639,...,-0.652835,-0.022868,-0.240003,-0.687651,-1.127053,0.614484,1.767473,0.202531,-1.030579,0.618495
2,-1.201331,-1.831983,1.410737,1.402652,1.499806,1.098857,0.623053,-0.741007,-0.980205,-0.988639,...,-0.610901,-0.023076,-0.240003,-0.687651,-1.127053,0.614484,1.636944,-0.139061,-0.895674,0.588118
3,-1.201331,-1.831983,1.410737,-1.122731,1.517073,-1.249485,0.848464,-0.782027,-1.019814,-1.041115,...,-0.564556,-0.022868,-0.240003,-0.763333,-0.890036,0.813350,1.767473,-0.538009,-1.030579,1.054491
4,-1.216034,-1.694780,0.963239,1.056581,1.510166,1.539171,0.623053,-0.850394,-1.019814,-1.014877,...,-0.690779,-0.023020,-0.240003,-0.687651,-1.127053,1.211083,1.636944,-0.536015,-1.089473,1.285053


# Models

## Basic model

In [783]:
models = {}
for idx, col in enumerate(y_pred_columns):
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X, y.iloc[:, idx])
    y_pred.iloc[:, idx] = model.predict(X_pred)
    models[col] = model
    print(col, y.iloc[:, idx])

KeyboardInterrupt: 

In [ ]:
print(y.shape)
y.head()

(44298, 12)


,Al,B,Ca,Cu,Fe,K,Mg,Mn,Na,P,S,Zn
0,1109.856,0.0,1535.388,2.259,92.366,283.103,200.601,107.257,37.98841,0.0,0.0,0.0
1,1168.364,0.0,751.408,1.822,115.923,215.459,197.771,90.005,37.98841,0.0,0.0,0.0
2,1137.113,0.0,468.391,1.913,78.709,398.656,188.114,120.433,37.98841,0.0,0.0,0.0
3,1117.349,0.0,739.698,2.876,127.527,267.354,156.417,112.036,37.98841,0.0,0.0,0.0
4,1219.203,0.0,240.071,1.825,77.542,229.682,114.809,57.906,37.98841,0.0,0.0,0.0


In [ ]:
print(y_pred.shape)
y_pred.head()

(6070, 13)


,Target_Al,Target_B,Target_Ca,Target_Cu,Target_Fe,Target_K,Target_Mg,Target_Mn,Target_N,Target_Na,Target_P,Target_S,Target_Zn
0,1080.811740,0.0,1603.482045,3.920577,114.626471,424.380146,632.948935,155.185500,3.005322,37.776041,0.0,0.0,0.0
1,835.471390,0.0,1152.423855,3.334555,128.250410,181.475743,271.124675,156.168038,2.346867,37.882225,0.0,0.0,0.0
2,1204.225790,0.0,1263.695854,3.359850,172.421600,180.035940,292.210300,185.226725,2.019400,37.895154,0.0,0.0,0.0
3,1028.011037,0.0,1660.373240,3.046398,186.119812,369.832900,292.490400,103.661233,2.972250,37.988410,0.0,0.0,0.0
4,598.950005,0.0,1521.663768,3.357210,113.557549,298.824980,313.862635,157.284172,1.169396,37.988410,0.0,0.0,0.0


In [ ]:
olek_features.head()

,ID,Country,Area,tmin_avg,tmax_avg,prec_avg,B11_x,B12_x,B2,B3,...,Fe_pred,K_pred,Mg_pred,N,P_pred,S_pred,Zn_pred,carbon,clay,ph
0,O2TONM,Tanzania,Tanzania,12.727273,22.628788,123.26878,1287.0,699.0,243.0,364.0,...,28.964100,133.289780,120.510418,1.208738e+07,8.974182,2.320117,0.822119,32.0,36.0,5.6
1,BQLUK6,Tanzania,Tanzania,12.727273,22.628788,123.26878,3215.0,2426.0,789.0,1066.0,...,26.112639,133.289780,108.947172,2.979958e+06,7.166170,2.004166,0.648721,30.0,40.0,5.7
2,LSET8M,Tanzania,Tanzania,12.727273,22.628788,123.26878,3127.0,2220.0,697.0,943.0,...,28.964100,133.289780,120.510418,1.634984e+06,7.166170,2.004166,0.648721,30.0,39.0,5.7
3,LEEL7I,Tanzania,Tanzania,12.727273,22.628788,123.26878,2085.0,1370.0,499.0,691.0,...,26.112639,108.947172,133.289780,2.979958e+06,7.166170,1.718282,0.822119,31.0,40.0,5.5
4,LDNGO2,Tanzania,Tanzania,12.674242,23.077652,110.35551,2059.0,1076.0,324.0,436.0,...,23.532530,147.413159,98.484316,1.997196e+06,7.166170,2.004166,0.648721,33.0,39.0,5.0


In [ ]:
columns = y_pred.columns
missing_minerals = olek_features[['ID', 'S_pred', 'Zn_pred', 'P_pred']]
y_pred.merge(missing_minerals, on='ID', how='left')
y_pred.head()

KeyError: 'ID'

In [ ]:
id = test_df["ID"]
result = pd.concat([id, y_pred], axis=1)
result.head()


,ID,Target_Al,Target_B,Target_Ca,Target_Cu,Target_Fe,Target_K,Target_Mg,Target_Mn,Target_N,Target_Na,Target_P,Target_S,Target_Zn
0,8ZMJRO,1080.811740,0.0,1603.482045,3.920577,114.626471,424.380146,632.948935,155.185500,3.005322,37.776041,0.0,0.0,0.0
1,DCC6DM,835.471390,0.0,1152.423855,3.334555,128.250410,181.475743,271.124675,156.168038,2.346867,37.882225,0.0,0.0,0.0
2,T50LK1,1204.225790,0.0,1263.695854,3.359850,172.421600,180.035940,292.210300,185.226725,2.019400,37.895154,0.0,0.0,0.0
3,FNLYT0,1028.011037,0.0,1660.373240,3.046398,186.119812,369.832900,292.490400,103.661233,2.972250,37.988410,0.0,0.0,0.0
4,FP5E12,598.950005,0.0,1521.663768,3.357210,113.557549,298.824980,313.862635,157.284172,1.169396,37.988410,0.0,0.0,0.0


In [ ]:
pred_to_keep.shape

(6070, 14)

In [ ]:
pred_to_keep.columns = result.columns
pred_to_keep.head()

,ID,Target_Al,Target_B,Target_Ca,Target_Cu,Target_Fe,Target_K,Target_Mg,Target_Mn,Target_N,Target_Na,Target_P,Target_S,Target_Zn
0,00A83S,1,1,1,1,1,1,1,1,1,1,1,1,1
1,00F2Q3,1,1,1,1,1,1,1,1,1,1,1,1,1
2,00FNFP,1,0,1,1,1,1,1,1,1,0,0,0,0
3,01MFSS,1,0,1,1,1,1,1,1,1,0,0,0,0
4,02851F,1,0,1,1,1,1,1,1,1,0,0,0,0


In [ ]:
result_indexed = result.set_index("ID")
pred_indexed = pred_to_keep.set_index("ID")

cols = pred_indexed.columns
result_indexed[cols] = result_indexed[cols].where(pred_indexed[cols] == 1, other=0)

result = result_indexed.reset_index()
result.head()

,ID,Target_Al,Target_B,Target_Ca,Target_Cu,Target_Fe,Target_K,Target_Mg,Target_Mn,Target_N,Target_Na,Target_P,Target_S,Target_Zn
0,8ZMJRO,1540.223985,0.208071,16724.079227,4.233323,293.22949,1864.09500,2159.528746,210.646136,75.776177,24.720507,3.226488,0.583553,0.0
1,DCC6DM,1540.223985,0.208071,16711.674999,4.227703,293.22949,2031.40577,2160.196873,210.237806,77.961780,24.720507,3.226488,0.583553,0.0
2,T50LK1,1540.223985,0.208071,16711.674999,4.213542,293.22949,2031.40577,2160.196873,210.237806,77.961780,24.783627,3.226488,0.583553,0.0
3,FNLYT0,1540.223985,0.208071,16686.128239,4.212302,293.36433,1978.61073,2192.241223,209.036006,74.463220,24.677201,3.262684,0.583553,0.0
4,FP5E12,1556.552625,0.208071,16686.128239,4.212302,293.36433,1978.61073,2195.959629,209.036006,77.643333,24.677201,3.262684,0.583553,0.0


In [ ]:
result.to_csv("../rhea-soil-nutrient-prediction-challenge/submissions/first-minerals-submission.csv", index=False)

## Rfr with tunning

In [660]:
from sklearn.model_selection import RandomizedSearchCV

In [661]:
param_grid = {
    'n_estimators': [100, 300, 500],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', 1.0] 
}

In [664]:
for idx, col in enumerate(y_pred_columns):
    print(f"Param tunning for: {col}...")
    rf = RandomForestRegressor(random_state=42)
    
    search = RandomizedSearchCV(
        estimator=rf, 
        param_distributions=param_grid, 
        n_iter=10,
        cv=3, 
        scoring='neg_root_mean_squared_error', 
        random_state=42, 
        n_jobs=-1
    )
    
    search.fit(X, y.iloc[:, idx])
    best_model = search.best_estimator_
    y_pred.iloc[:, idx] = best_model.predict(X_pred)
    models[col] = best_model
    best_rmse = -search.best_score_
    
    print("Best params: ", search.best_params_)
    print("Est. RMSE:" , best_rmse)

Param tunning for: Target_Al...
Best params:  {'n_estimators': 100, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': None}
Est. RMSE: 164.25829266375263
Param tunning for: Target_B...
Best params:  {'n_estimators': 100, 'min_samples_split': 10, 'min_samples_leaf': 4, 'max_features': 'log2', 'max_depth': 10}
Est. RMSE: 0.01886076365646781
Param tunning for: Target_Ca...
Best params:  {'n_estimators': 100, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 1.0, 'max_depth': 10}
Est. RMSE: 992.6192760658179
Param tunning for: Target_Cu...
Best params:  {'n_estimators': 100, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': None}
Est. RMSE: 0.8745007669125008
Param tunning for: Target_Fe...
Best params:  {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 4, 'max_features': 'sqrt', 'max_depth': 30}
Est. RMSE: 29.96075335209872
Param tunning for: Target_K...
Best params:  {'n_estimators': 100, 'mi

In [665]:
print(y_pred.shape)
y_pred.head()

(6070, 13)


,Target_Al,Target_B,Target_Ca,Target_Cu,Target_Fe,Target_K,Target_Mg,Target_Mn,Target_N,Target_Na,Target_P,Target_S,Target_Zn
0,1153.621620,0.000000,1586.421015,3.849069,126.997113,461.209326,533.275324,147.844989,2.961335,37.994839,0.000000,0.000338,0.003250
1,1046.541991,0.000232,1046.007247,2.972119,131.392864,191.044054,280.264972,138.552357,2.311901,37.969372,0.061800,0.000356,0.002227
2,1205.804563,0.000005,1198.579291,3.220352,163.734173,190.288420,274.227818,156.063478,2.201741,38.027427,0.061800,0.000356,0.005477
3,1185.685210,0.000000,1572.802890,2.785660,168.030371,358.316517,240.060982,121.815151,3.207623,37.988908,0.066628,0.000000,0.000000
4,737.028882,0.000000,1388.817216,3.013423,118.520028,295.023526,318.518817,154.517902,1.075048,37.988105,0.000000,0.000000,0.000027


In [666]:
id = test_df["ID"]
result = pd.concat([id, y_pred], axis=1)
result.head()

,ID,Target_Al,Target_B,Target_Ca,Target_Cu,Target_Fe,Target_K,Target_Mg,Target_Mn,Target_N,Target_Na,Target_P,Target_S,Target_Zn
0,8ZMJRO,1153.621620,0.000000,1586.421015,3.849069,126.997113,461.209326,533.275324,147.844989,2.961335,37.994839,0.000000,0.000338,0.003250
1,DCC6DM,1046.541991,0.000232,1046.007247,2.972119,131.392864,191.044054,280.264972,138.552357,2.311901,37.969372,0.061800,0.000356,0.002227
2,T50LK1,1205.804563,0.000005,1198.579291,3.220352,163.734173,190.288420,274.227818,156.063478,2.201741,38.027427,0.061800,0.000356,0.005477
3,FNLYT0,1185.685210,0.000000,1572.802890,2.785660,168.030371,358.316517,240.060982,121.815151,3.207623,37.988908,0.066628,0.000000,0.000000
4,FP5E12,737.028882,0.000000,1388.817216,3.013423,118.520028,295.023526,318.518817,154.517902,1.075048,37.988105,0.000000,0.000000,0.000027


## XGB

In [520]:
X.head()

,tmin_avg,tmax_avg,prec_avg,aspect,elevation,hillshade,slope,cec_0-5cm_mean,cec_15-30cm_mean,cec_30-60cm_mean,...,Fe_pred,K_pred,Mg_pred,N_pred,P_pred,S_pred,Zn_pred,carbon,clay,ph
0,-1.201331,-1.831983,1.410737,0.252200,1.511893,0.658543,1.975520,-0.741007,-0.967002,-0.988639,...,-0.787508,-0.278184,-0.610901,-0.021464,0.480508,-0.604010,-0.890036,1.012217,1.245357,-0.880190
1,-1.201331,-1.831983,1.410737,0.934989,1.492899,2.566571,1.524698,-0.741007,-0.980205,-0.988639,...,-0.889023,-0.278184,-0.652835,-0.022868,-0.240003,-0.687651,-1.127053,0.614484,1.767473,-0.745993
2,-1.201331,-1.831983,1.410737,1.402652,1.499806,1.098857,0.623053,-0.741007,-0.980205,-0.988639,...,-0.787508,-0.278184,-0.610901,-0.023076,-0.240003,-0.687651,-1.127053,0.614484,1.636944,-0.745993
3,-1.201331,-1.831983,1.410737,-1.122731,1.517073,-1.249485,0.848464,-0.782027,-1.019814,-1.041115,...,-0.889023,-0.443964,-0.564556,-0.022868,-0.240003,-0.763333,-0.890036,0.813350,1.767473,-1.014387
4,-1.216034,-1.694780,0.963239,1.056581,1.510166,1.539171,0.623053,-0.850394,-1.019814,-1.014877,...,-0.980878,-0.182000,-0.690779,-0.023020,-0.240003,-0.687651,-1.127053,1.211083,1.636944,-1.685372


In [521]:
X_pred.head()

,tmin_avg,tmax_avg,prec_avg,aspect,elevation,hillshade,slope,cec_0-5cm_mean,cec_15-30cm_mean,cec_30-60cm_mean,...,Fe_pred,K_pred,Mg_pred,N_pred,P_pred,S_pred,Zn_pred,carbon,clay,ph
0,-0.697771,-0.805570,0.915603,-0.954372,0.660630,-0.809171,-0.053180,0.325518,0.472112,0.297032,...,0.519838,1.056700,0.411483,-0.019573,0.898543,0.906234,8.955596,2.205415,2.028531,-0.477599
1,-0.411073,-0.404962,0.327047,-1.178850,0.296296,-2.790584,3.553398,0.694699,0.656952,0.611890,...,0.769524,-0.075699,-0.248626,-0.021947,1.871131,0.678874,4.101646,1.608816,2.159060,-0.880190
2,-0.820117,-0.947403,1.143224,1.243647,0.665810,4.694755,4.455042,1.022861,0.788981,0.743081,...,1.350437,-0.075699,-0.248626,-0.014093,1.871131,1.157505,3.389608,1.211083,1.375886,-0.880190
3,-1.425544,-1.391430,2.877642,0.074488,1.763991,-0.222085,0.623053,0.448578,0.313677,0.152722,...,1.687478,0.473687,-0.394173,0.060019,1.871131,-0.296510,3.389608,1.807682,1.636944,-1.014387
4,-0.843746,-1.051608,-1.285319,1.402652,0.919635,0.878700,0.172231,0.612659,0.828590,0.874272,...,0.769524,0.473687,-0.070853,-0.023203,3.748276,-0.171732,0.334929,0.216751,0.331654,0.193386


In [522]:
y.head()

,Al,B,Ca,Cu,Fe,K,Mg,Mn,N,Na,P,S,Zn
0,1109.856,0.0,1535.388,2.259,92.366,283.103,200.601,107.257,2.24,37.98841,0.0,0.0,0.0
1,1168.364,0.0,751.408,1.822,115.923,215.459,197.771,90.005,1.57,37.98841,0.0,0.0,0.0
2,1137.113,0.0,468.391,1.913,78.709,398.656,188.114,120.433,1.02,37.98841,0.0,0.0,0.0
3,1117.349,0.0,739.698,2.876,127.527,267.354,156.417,112.036,1.12,37.98841,0.0,0.0,0.0
4,1219.203,0.0,240.071,1.825,77.542,229.682,114.809,57.906,1.19,37.98841,0.0,0.0,0.0


In [464]:
y.info()

<class 'pandas.DataFrame'>
RangeIndex: 44298 entries, 0 to 44297
Data columns (total 13 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Al      44296 non-null  float64
 1   B       1909 non-null   float64
 2   Ca      44298 non-null  float64
 3   Cu      44257 non-null  float64
 4   Fe      44257 non-null  float64
 5   K       44298 non-null  float64
 6   Mg      44298 non-null  float64
 7   Mn      44255 non-null  float64
 8   N       44257 non-null  float64
 9   Na      44298 non-null  float64
 10  P       1909 non-null   float64
 11  S       1909 non-null   float64
 12  Zn      1909 non-null   float64
dtypes: float64(13)
memory usage: 4.4 MB


In [523]:
models = {}
for idx, col in enumerate(y_pred_columns):
    model = XGBRegressor(n_estimators=1000, random_state=42)
    model.fit(X, y.iloc[:, idx], verbose=False)
    y_pred.iloc[:, idx] = model.predict(X_pred)
    models[col] = model
    print(col, y.iloc[:, idx])

Target_Al 0        1109.856
1        1168.364
2        1137.113
3        1117.349
4        1219.203
           ...   
44293    1056.900
44294     320.713
44295     452.637
44296       0.000
44297       0.000
Name: Al, Length: 44298, dtype: float64
Target_B 0        0.000000
1        0.000000
2        0.000000
3        0.000000
4        0.000000
           ...   
44293    0.009823
44294    0.009902
44295    0.009937
44296    0.360000
44297    0.380000
Name: B, Length: 44298, dtype: float64
Target_Ca 0        1535.388
1         751.408
2         468.391
3         739.698
4         240.071
           ...   
44293     692.423
44294     475.510
44295     318.088
44296    1390.000
44297    1430.000
Name: Ca, Length: 44298, dtype: float64
Target_Cu 0        2.259000
1        1.822000
2        1.913000
3        2.876000
4        1.825000
           ...   
44293    1.162110
44294    0.095523
44295    0.462887
44296    2.220000
44297    2.430000
Name: Cu, Length: 44298, dtype: float64
Target_Fe 

In [524]:
print(y_pred.shape)
y_pred.head()

(6070, 13)


,Target_Al,Target_B,Target_Ca,Target_Cu,Target_Fe,Target_K,Target_Mg,Target_Mn,Target_N,Target_Na,Target_P,Target_S,Target_Zn
0,1283.934204,-0.000098,1275.911255,4.250233,96.509979,390.601074,639.487244,164.052048,2.442327,33.411755,-0.036390,-0.006553,0.000683
1,855.664551,-0.000260,1442.394287,3.040687,126.535728,226.545578,528.830933,152.324173,2.170465,33.278355,-0.005042,0.124563,0.011395
2,1051.225952,-0.000114,2431.542969,2.638163,139.827133,103.747185,386.384888,105.869560,1.339506,37.855663,0.016653,-0.071148,-0.004484
3,916.150269,-0.001385,1996.280151,1.068102,163.798889,380.795380,416.641052,51.496292,3.047634,36.116348,-0.127875,0.007731,-0.003987
4,594.226379,-0.000135,1266.052246,3.159769,113.616875,376.489288,249.230621,162.154510,1.098050,37.976238,-0.001505,0.003396,0.000131


In [144]:
id = test_df["ID"]
result = pd.concat([id, y_pred], axis=1)
result.head()

,ID,Target_Al,Target_B,Target_Ca,Target_Cu,Target_Fe,Target_K,Target_Mg,Target_Mn,Target_N,Target_Na,Target_P,Target_S,Target_Zn
0,8ZMJRO,1142.374634,0.0,2812.468506,3.870714,159.949005,165.705307,632.682739,224.051865,2.644720,32.129364,0.0,0.0,0.0
1,DCC6DM,896.380066,0.0,2844.757812,3.326280,185.121429,376.345825,657.277771,170.576111,1.793384,32.158688,0.0,0.0,0.0
2,T50LK1,1268.962646,0.0,2728.878174,2.330326,166.559326,316.379700,737.470642,221.666031,3.308956,31.685741,0.0,0.0,0.0
3,FNLYT0,1337.523315,0.0,2620.776855,3.228681,178.334564,576.343567,832.497559,226.330872,2.938833,31.925537,0.0,0.0,0.0
4,FP5E12,595.373230,0.0,3361.533447,4.042684,112.238197,620.369995,856.571045,151.752731,0.887873,32.148026,0.0,0.0,0.0


In [429]:
pred_to_keep.columns = result.columns
pred_to_keep.head()

,ID,Target_Al,Target_B,Target_Ca,Target_Cu,Target_Fe,Target_K,Target_Mg,Target_Mn,Target_N,Target_Na,Target_P,Target_S,Target_Zn
0,00A83S,1,1,1,1,1,1,1,1,1,1,1,1,1
1,00F2Q3,1,1,1,1,1,1,1,1,1,1,1,1,1
2,00FNFP,1,0,1,1,1,1,1,1,1,0,0,0,0
3,01MFSS,1,0,1,1,1,1,1,1,1,0,0,0,0
4,02851F,1,0,1,1,1,1,1,1,1,0,0,0,0


In [146]:
result_indexed = result.set_index("ID")
pred_indexed = pred_to_keep.set_index("ID")

cols = pred_indexed.columns
result_indexed[cols] = result_indexed[cols].where(pred_indexed[cols] == 1, other=0)

result = result_indexed.reset_index()
result.head()

,ID,Target_Al,Target_B,Target_Ca,Target_Cu,Target_Fe,Target_K,Target_Mg,Target_Mn,Target_N,Target_Na,Target_P,Target_S,Target_Zn
0,8ZMJRO,1142.374634,0.0,2812.468506,3.870714,159.949005,165.705307,632.682739,224.051865,2.644720,32.129364,0.0,0.0,0.0
1,DCC6DM,896.380066,0.0,2844.757812,3.326280,185.121429,376.345825,657.277771,170.576111,1.793384,32.158688,0.0,0.0,0.0
2,T50LK1,1268.962646,0.0,2728.878174,2.330326,166.559326,316.379700,737.470642,221.666031,3.308956,31.685741,0.0,0.0,0.0
3,FNLYT0,1337.523315,0.0,2620.776855,3.228681,178.334564,576.343567,832.497559,226.330872,2.938833,31.925537,0.0,0.0,0.0
4,FP5E12,595.373230,0.0,3361.533447,4.042684,112.238197,620.369995,856.571045,151.752731,0.887873,32.148026,0.0,0.0,0.0


In [147]:
result.to_csv("../rhea-soil-nutrient-prediction-challenge/submissions/first-XGBoost-submission.csv", index=False)

In [557]:
X.columns

Index(['tmin_avg', 'tmax_avg', 'prec_avg', 'aspect', 'elevation', 'hillshade',
       'slope', 'cec_0-5cm_mean', 'cec_15-30cm_mean', 'cec_30-60cm_mean',
       'cec_5-15cm_mean', 'clay_0-5cm_mean', 'clay_15-30cm_mean',
       'clay_30-60cm_mean', 'clay_5-15cm_mean', 'phh2o_0-5cm_mean',
       'phh2o_15-30cm_mean', 'phh2o_30-60cm_mean', 'phh2o_5-15cm_mean',
       'sand_0-5cm_mean', 'sand_15-30cm_mean', 'sand_30-60cm_mean',
       'sand_5-15cm_mean', 'soc_0-5cm_mean', 'soc_15-30cm_mean',
       'soc_30-60cm_mean', 'soc_5-15cm_mean', 'B04', 'B05', 'B06', 'B07',
       'B08', 'B09', 'B10', 'B11_y', 'B12_y', 'B13', 'B14', 'Al_pred',
       'Ca_pred', 'Fe_pred', 'K_pred', 'Mg_pred', 'N_pred', 'P_pred', 'S_pred',
       'Zn_pred', 'carbon', 'clay', 'ph'],
      dtype='str')

In [558]:
listed_elements = ['Al_pred','Ca_pred', 'Fe_pred', 'K_pred', 'Mg_pred', 'N_pred', 'P_pred', 'S_pred']

## XGB CV

In [1058]:
targets = y_columns 

models = {}
oof_preds = pd.DataFrame(index=X.index, columns=targets)
test_preds = pd.DataFrame(index=X_pred.index, columns=targets)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for target_name in targets:
    print(f"--- Training model for: {target_name} ---")
    y_target = y[target_name]
    
    if y_target.nunique() <= 1:
        print(f"{target_name} constant value. Returning 0.")
        oof_preds[target_name] = 0
        test_preds[target_name] = 0
        continue

    target_test_preds = np.zeros(len(X_pred))
    fold_scores = []
    
    #applying log transformation
    y_target_log = np.log1p(y_target)

    for fold, (train_idx, valid_idx) in enumerate(kf.split(X, y_target_log)):
        X_train_fold, X_valid_fold = X.iloc[train_idx], X.iloc[valid_idx]
        y_train_fold, y_valid_fold = y_target_log.iloc[train_idx], y_target_log.iloc[valid_idx]
        
        # 1st opt: n = 2000, lr = 0.03, esr = 150
        # 2nd opt: n = 5000, lr = 0.01, esr = 300
        model = XGBRegressor(
            n_estimators=2000,        
            learning_rate=0.03,       
            max_depth=5,              
            subsample=0.8,            
            colsample_bytree=0.8,     
            random_state=42 + fold,
            early_stopping_rounds=150, 
            n_jobs=-1
        )
        
        model.fit(
            X_train_fold, y_train_fold,
            eval_set=[(X_valid_fold, y_valid_fold)],
            verbose=False
        )
        
        #returning to normal scale
        valid_pred_log = model.predict(X_valid_fold)
        valid_pred = np.expm1(valid_pred_log)

        oof_preds.loc[valid_idx, target_name] = valid_pred
        
        rmse = root_mean_squared_error(y_valid_fold, valid_pred)
        fold_scores.append(rmse)
        
        #returning to normal scale
        target_test_preds += np.expm1(model.predict(X_pred)) / kf.n_splits
        
    print(f"Average RMSE for {target_name}: {np.mean(fold_scores):.4f}")
    
    test_preds[target_name] = target_test_preds

result = pd.DataFrame({'ID': test_df['ID']})
test_preds.columns = [f"Target_{col}" for col in test_preds.columns]
result = pd.concat([result, test_preds], axis=1)


--- Training model for: Al ---
Average RMSE for Al: 832.5854
--- Training model for: B ---
Average RMSE for B: 0.0106
--- Training model for: Ca ---
Average RMSE for Ca: 2967.5807
--- Training model for: Cu ---
Average RMSE for Cu: 1.3982
--- Training model for: Fe ---
Average RMSE for Fe: 107.7668
--- Training model for: K ---
Average RMSE for K: 301.9791
--- Training model for: Mg ---
Average RMSE for Mg: 469.2000
--- Training model for: Mn ---
Average RMSE for Mn: 124.3657
--- Training model for: N ---
Average RMSE for N: 0.5412
--- Training model for: Na ---
Average RMSE for Na: 34.2525
--- Training model for: P ---
Average RMSE for P: 0.9474
--- Training model for: S ---
Average RMSE for S: 1.1786
--- Training model for: Zn ---
Average RMSE for Zn: 0.0981


In [1059]:
y.head()

,Al,B,Ca,Cu,Fe,K,Mg,Mn,N,Na,P,S,Zn
0,1109.856,0.0,1535.388,2.259,92.366,283.103,200.601,107.257,2.24,37.98841,0.0,0.0,0.0
1,1168.364,0.0,751.408,1.822,115.923,215.459,197.771,90.005,1.57,37.98841,0.0,0.0,0.0
2,1137.113,0.0,468.391,1.913,78.709,398.656,188.114,120.433,1.02,37.98841,0.0,0.0,0.0
3,1117.349,0.0,739.698,2.876,127.527,267.354,156.417,112.036,1.12,37.98841,0.0,0.0,0.0
4,1219.203,0.0,240.071,1.825,77.542,229.682,114.809,57.906,1.19,37.98841,0.0,0.0,0.0


In [1060]:
result.head()

,ID,Target_Al,Target_B,Target_Ca,Target_Cu,Target_Fe,Target_K,Target_Mg,Target_Mn,Target_N,Target_Na,Target_P,Target_S,Target_Zn
0,8ZMJRO,1115.987701,0.000124,1556.419800,3.898888,110.898418,449.685509,476.088646,121.817842,2.517077,38.065728,-0.000001,0.002296,-0.000152
1,DCC6DM,841.715546,-0.000002,935.815109,2.678792,113.086283,172.934048,257.461891,113.245428,1.816288,37.980696,0.003776,-0.002789,-0.000944
2,T50LK1,1023.306366,0.000606,1153.525146,2.787807,139.169241,155.119110,291.677204,105.489777,1.668604,37.989917,0.012603,0.072787,0.011314
3,FNLYT0,779.938553,-0.000254,1450.033997,2.250226,144.509066,385.589828,276.288544,66.758653,3.410547,37.724824,-0.002780,0.001343,-0.002315
4,FP5E12,642.186783,-0.000341,1133.058136,2.127377,105.751272,291.481426,287.432911,115.407267,1.163343,37.949699,-0.003061,-0.001108,-0.002070


# Adding missing elements

### Joining minerals for columns with missing data

In [1061]:
result_mod = result.copy()

In [1062]:
missing_minerals = ['Target_B', 'Target_P', 'Target_S', 'Target_Zn']
for mineral in missing_minerals:
    result_mod[mineral] = result_mod[mineral].clip(lower=0)
result_mod.head()

,ID,Target_Al,Target_B,Target_Ca,Target_Cu,Target_Fe,Target_K,Target_Mg,Target_Mn,Target_N,Target_Na,Target_P,Target_S,Target_Zn
0,8ZMJRO,1115.987701,0.000124,1556.419800,3.898888,110.898418,449.685509,476.088646,121.817842,2.517077,38.065728,0.000000,0.002296,0.000000
1,DCC6DM,841.715546,0.000000,935.815109,2.678792,113.086283,172.934048,257.461891,113.245428,1.816288,37.980696,0.003776,0.000000,0.000000
2,T50LK1,1023.306366,0.000606,1153.525146,2.787807,139.169241,155.119110,291.677204,105.489777,1.668604,37.989917,0.012603,0.072787,0.011314
3,FNLYT0,779.938553,0.000000,1450.033997,2.250226,144.509066,385.589828,276.288544,66.758653,3.410547,37.724824,0.000000,0.001343,0.000000
4,FP5E12,642.186783,0.000000,1133.058136,2.127377,105.751272,291.481426,287.432911,115.407267,1.163343,37.949699,0.000000,0.000000,0.000000


In [1063]:
y[['B', 'P', 'S', 'Zn']].describe()

,B,P,S,Zn
count,44298.000000,44298.000000,44298.000000,44298.000000
mean,0.002899,0.267682,0.336817,0.054154
std,0.020403,1.599019,1.764488,0.267449
min,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.000000,0.000000
max,0.705266,83.700000,44.533500,2.771600


In [1064]:
mis_min = ['P_pred', 'S_pred', 'Zn_pred']
missing_minerals_pred = olek_features[['ID'] + mis_min]
missing_minerals_pred.head()

,ID,P_pred,S_pred,Zn_pred
0,O2TONM,8.974182,2.320117,0.822119
1,BQLUK6,7.166170,2.004166,0.648721
2,LSET8M,7.166170,2.004166,0.648721
3,LEEL7I,7.166170,1.718282,0.822119
4,LDNGO2,7.166170,2.004166,0.648721


In [1065]:
result_final = result_mod.merge(missing_minerals_pred, on='ID', how='left')

In [1066]:
result_final.drop(columns=['Target_P', 'Target_S', 'Target_Zn'], inplace=True)
result_final.head()

,ID,Target_Al,Target_B,Target_Ca,Target_Cu,Target_Fe,Target_K,Target_Mg,Target_Mn,Target_N,Target_Na,P_pred,S_pred,Zn_pred
0,8ZMJRO,1115.987701,0.000124,1556.419800,3.898888,110.898418,449.685509,476.088646,121.817842,2.517077,38.065728,10.023176,8.025013,8.025013
1,DCC6DM,841.715546,0.000000,935.815109,2.678792,113.086283,172.934048,257.461891,113.245428,1.816288,37.980696,12.463738,7.166170,4.473947
2,T50LK1,1023.306366,0.000606,1153.525146,2.787807,139.169241,155.119110,291.677204,105.489777,1.668604,37.989917,12.463738,8.974182,3.953032
3,FNLYT0,779.938553,0.000000,1450.033997,2.250226,144.509066,385.589828,276.288544,66.758653,3.410547,37.724824,12.463738,3.481689,3.953032
4,FP5E12,642.186783,0.000000,1133.058136,2.127377,105.751272,291.481426,287.432911,115.407267,1.163343,37.949699,17.174145,3.953032,1.718282


In [1067]:
result_final.columns = result_final.columns[:-3].tolist() + ['Target_P', 'Target_S', 'Target_Zn']
result_final.head()

,ID,Target_Al,Target_B,Target_Ca,Target_Cu,Target_Fe,Target_K,Target_Mg,Target_Mn,Target_N,Target_Na,Target_P,Target_S,Target_Zn
0,8ZMJRO,1115.987701,0.000124,1556.419800,3.898888,110.898418,449.685509,476.088646,121.817842,2.517077,38.065728,10.023176,8.025013,8.025013
1,DCC6DM,841.715546,0.000000,935.815109,2.678792,113.086283,172.934048,257.461891,113.245428,1.816288,37.980696,12.463738,7.166170,4.473947
2,T50LK1,1023.306366,0.000606,1153.525146,2.787807,139.169241,155.119110,291.677204,105.489777,1.668604,37.989917,12.463738,8.974182,3.953032
3,FNLYT0,779.938553,0.000000,1450.033997,2.250226,144.509066,385.589828,276.288544,66.758653,3.410547,37.724824,12.463738,3.481689,3.953032
4,FP5E12,642.186783,0.000000,1133.058136,2.127377,105.751272,291.481426,287.432911,115.407267,1.163343,37.949699,17.174145,3.953032,1.718282


In [1068]:
order = ['ID','Target_Al', 'Target_B', 'Target_Ca', 'Target_Cu', 'Target_Fe', 'Target_K', 'Target_Mg', 'Target_Mn', 'Target_N', 'Target_Na', 'Target_P', 'Target_S', 'Target_Zn']

In [1069]:
result_final = result_final[order]
result_final.head()

,ID,Target_Al,Target_B,Target_Ca,Target_Cu,Target_Fe,Target_K,Target_Mg,Target_Mn,Target_N,Target_Na,Target_P,Target_S,Target_Zn
0,8ZMJRO,1115.987701,0.000124,1556.419800,3.898888,110.898418,449.685509,476.088646,121.817842,2.517077,38.065728,10.023176,8.025013,8.025013
1,DCC6DM,841.715546,0.000000,935.815109,2.678792,113.086283,172.934048,257.461891,113.245428,1.816288,37.980696,12.463738,7.166170,4.473947
2,T50LK1,1023.306366,0.000606,1153.525146,2.787807,139.169241,155.119110,291.677204,105.489777,1.668604,37.989917,12.463738,8.974182,3.953032
3,FNLYT0,779.938553,0.000000,1450.033997,2.250226,144.509066,385.589828,276.288544,66.758653,3.410547,37.724824,12.463738,3.481689,3.953032
4,FP5E12,642.186783,0.000000,1133.058136,2.127377,105.751272,291.481426,287.432911,115.407267,1.163343,37.949699,17.174145,3.953032,1.718282


### Removing predictions for samples without mineral data 

In [1070]:
pred_to_keep.columns

Index(['ID', 'Al', 'B', 'Ca', 'Cu', 'Fe', 'K', 'Mg', 'Mn', 'N', 'Na', 'P', 'S',
       'Zn'],
      dtype='str')

In [1071]:
pred_to_keep.columns = result_final.columns
pred_to_keep.head()

result_indexed = result_final.set_index("ID")
pred_indexed = pred_to_keep.set_index("ID")

cols = pred_indexed.columns
result_indexed[cols] = result_indexed[cols].where(pred_indexed[cols] == 1, other=0)

result_final = result_indexed.reset_index()
result_final.head()

,ID,Target_Al,Target_B,Target_Ca,Target_Cu,Target_Fe,Target_K,Target_Mg,Target_Mn,Target_N,Target_Na,Target_P,Target_S,Target_Zn
0,8ZMJRO,1115.987701,0.000124,1556.419800,3.898888,110.898418,449.685509,476.088646,121.817842,2.517077,38.065728,10.023176,8.025013,8.025013
1,DCC6DM,841.715546,0.000000,935.815109,2.678792,113.086283,172.934048,257.461891,113.245428,1.816288,37.980696,12.463738,7.166170,4.473947
2,T50LK1,1023.306366,0.000606,1153.525146,2.787807,139.169241,155.119110,291.677204,105.489777,1.668604,37.989917,12.463738,8.974182,3.953032
3,FNLYT0,779.938553,0.000000,1450.033997,2.250226,144.509066,385.589828,276.288544,66.758653,3.410547,37.724824,12.463738,3.481689,3.953032
4,FP5E12,642.186783,0.000000,1133.058136,2.127377,105.751272,291.481426,287.432911,115.407267,1.163343,37.949699,17.174145,3.953032,1.718282


In [1072]:
X.columns

Index(['tmin_avg', 'tmax_avg', 'prec_avg', 'aspect', 'elevation', 'hillshade',
       'slope', 'cec_0-5cm_mean', 'cec_15-30cm_mean', 'cec_30-60cm_mean',
       'cec_5-15cm_mean', 'clay_0-5cm_mean', 'clay_15-30cm_mean',
       'clay_30-60cm_mean', 'clay_5-15cm_mean', 'phh2o_0-5cm_mean',
       'phh2o_15-30cm_mean', 'phh2o_30-60cm_mean', 'phh2o_5-15cm_mean',
       'sand_0-5cm_mean', 'sand_15-30cm_mean', 'sand_30-60cm_mean',
       'sand_5-15cm_mean', 'soc_0-5cm_mean', 'soc_15-30cm_mean',
       'soc_30-60cm_mean', 'soc_5-15cm_mean', 'B04', 'B05', 'B06', 'B07',
       'B08', 'B09', 'B10', 'B11_y', 'B12_y', 'B13', 'B14', 'Al_pred',
       'Ca_pred', 'Fe_pred', 'K_pred', 'Mg_pred', 'N_pred', 'P_pred', 'S_pred',
       'Zn_pred', 'carbon', 'clay', 'ratio_Ca_Mg', 'ratio_Fe_Al', 'CIA_proxy'],
      dtype='str')

# Saving file for submission

In [1073]:
result_final.head()

,ID,Target_Al,Target_B,Target_Ca,Target_Cu,Target_Fe,Target_K,Target_Mg,Target_Mn,Target_N,Target_Na,Target_P,Target_S,Target_Zn
0,8ZMJRO,1115.987701,0.000124,1556.419800,3.898888,110.898418,449.685509,476.088646,121.817842,2.517077,38.065728,10.023176,8.025013,8.025013
1,DCC6DM,841.715546,0.000000,935.815109,2.678792,113.086283,172.934048,257.461891,113.245428,1.816288,37.980696,12.463738,7.166170,4.473947
2,T50LK1,1023.306366,0.000606,1153.525146,2.787807,139.169241,155.119110,291.677204,105.489777,1.668604,37.989917,12.463738,8.974182,3.953032
3,FNLYT0,779.938553,0.000000,1450.033997,2.250226,144.509066,385.589828,276.288544,66.758653,3.410547,37.724824,12.463738,3.481689,3.953032
4,FP5E12,642.186783,0.000000,1133.058136,2.127377,105.751272,291.481426,287.432911,115.407267,1.163343,37.949699,17.174145,3.953032,1.718282


In [1074]:
result_final.to_csv("../rhea-soil-nutrient-prediction-challenge/submissions/combined-features-XGB-log-150-submission.csv", index=False)